# Byte-Level BPE — Solution

Reference implementation for `02_byte_level_bpe_exercise.ipynb`.

## Setup

In [9]:
from collections import Counter

## Solution 1 — `pre_tokenize_bytes`

In [10]:
def pre_tokenize_bytes(corpus):
    byte_corpus = [tuple(word.encode("utf-8")) for sentence in corpus for word in sentence.lower().split()]
    count = Counter(byte_corpus)

    return count

In [11]:
# Sanity check
freqs = pre_tokenize_bytes(["hi", "hi there", "नमस्ते"])
for word, freq in freqs.items():
    print(word, '→', freq)
# You should see ints in [0, 256) for all the chars.
# Hindi entries will have higher byte values (>= 224) due to UTF-8 encoding

(104, 105) → 2
(116, 104, 101, 114, 101) → 1
(224, 164, 168, 224, 164, 174, 224, 164, 184, 224, 165, 141, 224, 164, 164, 224, 165, 135) → 1


**Why bytes, not characters?**  256 fixed values cover ANY Unicode text. The base vocabulary doesn't depend on what languages you support. The cost: non-ASCII text starts longer (3-4 bytes per character), so more merges are needed to compress it.

## Solution 2 — `ByteLevelBPETokenizer`

In [ ]:
def get_pair_counts(word_freqs: dict[tuple[str, ...], int]) -> dict[tuple[str, str], int]:
    pair_counts = Counter()

    for word,freq in word_freqs.items():
        for pair in zip(word, word[1:]):
            pair_counts[pair] += freq

    return pair_counts

In [ ]:
def merge_pair(
    pair: tuple[str, str],
    id,
    word_freqs: dict[tuple[str, ...], int],
) -> dict[tuple[str, ...], int]:
    updated_word_frequency = {}

    for word,freq in word_freqs.items():
        
        i = 0 
        new_word = []
        while i < len(word):
            if i < len(word)-1 and (word[i],word[i+1]) == pair:
                new_word.append(id)
                i += 2
            else:
                new_word.append(word[i])
                i +=1

        updated_word_frequency[tuple(new_word)] = freq

    return updated_word_frequency

In [15]:
class ByteLevelBPETokenizer:
    def __init__(self):
        # merges: list of ((a_id, b_id), new_id) — replayed in order at encode time
        self.merges = []
        # vocab: id -> bytes. Seeded with 256 single-byte entries so we can
        # always tokenize anything (every UTF-8 byte is in the vocab from step 0).
        self.vocab = {i: bytes([i]) for i in range(256)}

    def train(self, corpus, vocab_size):
        word_freqs = pre_tokenize_bytes(corpus)

        while len(self.vocab) < vocab_size:
            ## Count Pairs
            pair_counts = get_pair_counts(word_freqs)
            if not pair_counts:
                break
            
             ## Add the best pair to the merge to be used at encoding time
            best_pair = max(pair_counts, key=pair_counts.get)
            new_id = len(self.vocab)
            self.merges.append((best_pair, new_id))

            ### Reset word frequency
            word_freqs = merge_pair(best_pair, new_id, word_freqs)

            # The bytes for the new token are the concatenation of the pair's bytes.
            self.vocab[new_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
            

    def _apply_merge(self, tokens, pair, new_id):
        """Single-pass index walk: replace adjacent `pair` with `new_id`."""
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
                new_tokens.append(new_id)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        return tuple(new_tokens)


    def encode(self, text):
        ids = []
        for word in text.lower().split():
            tokens = tuple(word.encode('utf-8'))   # tuple of ints 0..255
            for pair, new_id in self.merges:
                tokens = self._apply_merge(tokens, pair, new_id)
            ids.extend(tokens)
        return ids

    def decode(self, ids):
        all_bytes = b''.join(self.vocab[i] for i in ids)
        return all_bytes.decode('utf-8', errors='replace')

**Why store the vocab as `id → bytes` (not `id → str`)?** Bytes don't always form valid UTF-8 in the middle of a merge. A merged token might span half a Devanagari character — that's not valid UTF-8 on its own. Keep everything as bytes until the very end (decode), where you concatenate all token bytes and call `.decode('utf-8')`.

**The decode trick:** to convert a list of token IDs back to text, look up each ID's bytes, concatenate them all, then `.decode('utf-8')` once at the end. NOT id-by-id — that would fail on multi-byte characters whose bytes got split across tokens.

## Solution 3 — Multilingual observation

In [16]:
# Train on a mostly-English corpus (the realistic scenario for a Western-built tokenizer).
corpus_en = [
    "the cat sat on the mat",
    "the dog ran across the field",
    "she opened the book and started to read",
    "machine learning is changing the world",
    "tokenization is the first step in nlp",
] * 50

tok = ByteLevelBPETokenizer()
tok.train(corpus_en, vocab_size=400)
print(f"vocab size: {len(tok.vocab)}, merges learned: {len(tok.merges)}\n")

# Same meaning ("I love machine learning"), four scripts.
sentences = {
    'English':  "I love machine learning",
    'Hindi':    "मुझे मशीन लर्निंग पसंद है",
    'Japanese': "私は機械学習が好きです",
    'Arabic':   "أحب تعلم الآلة",
}

print(f"{'Language':<10} {'chars':>6} {'utf8 bytes':>12} {'tokens':>8} {'chars/token':>14}")
print("-" * 55)
for lang, text in sentences.items():
    n_chars  = len(text)
    n_bytes  = len(text.encode('utf-8'))
    n_tokens = len(tok.encode(text))
    ratio    = n_chars / n_tokens if n_tokens else 0
    print(f"{lang:<10} {n_chars:>6} {n_bytes:>12} {n_tokens:>8} {ratio:>14.2f}")

# Expected pattern:
#   English wins on chars/token (merges learned for 'th', 'the', 'ing', etc.)
#   Hindi/Japanese/Arabic land near 0.3-0.5 chars/token — i.e. multiple tokens
#   *per character*, because each char is 2-3 UTF-8 bytes and no merges fire.
#   This is the "tokenizer tax" non-English speakers pay on every API call.

vocab size: 326, merges learned: 70

Language    chars   utf8 bytes   tokens    chars/token
-------------------------------------------------------
English        23           23        7           3.29
Hindi          25           67       63           0.40
Japanese       11           33       33           0.33
Arabic         14           26       24           0.58


**Why this matters.** this is the structural cause of multilingual inequity in LLMs. Even if the model is the same, the tokenizer charges 3-5x more tokens for non-Latin scripts. API costs, context windows, and inference speed all scale with token count._

## Run the tests

In [17]:
from tests import run_byte_level_bpe_tests
run_byte_level_bpe_tests(ByteLevelBPETokenizer)

Running Byte-Level BPE...
  ✓ initial vocab is exactly 256 bytes
  ✓ training grows vocab (within target)
  ✓ encode returns list of ints
  ✓ encode handles multilingual input

  All 4 checks passed ✓



True